# ShallowLandslider quickstart

This notebook demonstrates the Landlab-facing `ShallowLandslider` component on a small synthetic raster grid. It uses only Landlab APIs and synthetic fields, so it does not require an OpenTopography API key, local measured landslide files, or the standalone `EQShallowLandslider` utilities.

The workflow covers:

- building a grid and topographic field,
- creating flow-routing fields required for optional runout,
- adding soil depth and earthquake PGA fields,
- running `ShallowLandslider` in physics-only mode,
- plotting stability, selected landslides, and Newmark displacement,
- enabling the optional runout subcomponent and inspecting soil-depth change.


## Imports

In [ ]:
import matplotlib.pyplot as plt
import numpy as np

from landlab import RasterModelGrid
from landlab.components import PriorityFloodFlowRouter, ShallowLandslider

plt.style.use("seaborn-v0_8")
plt.rcParams.update({"figure.dpi": 120, "savefig.dpi": 300})

## Build a synthetic landscape

The example landscape is a gently sloping plane with small sinusoidal relief. This keeps the notebook self-contained while still creating realistic-looking slopes, aspect zones, selected landslide patches, and runout paths.


In [ ]:
nrows, ncols = 40, 50
spacing = 10.0
mg = RasterModelGrid((nrows, ncols), xy_spacing=spacing)

x = mg.node_x.reshape(mg.shape)
y = mg.node_y.reshape(mg.shape)
z_2d = 0.045 * (y.max() - y) + 0.010 * x + 2.0 * np.sin(x / 55.0) * np.cos(y / 70.0)

z = mg.add_field("topographic__elevation", z_2d.ravel(), at="node")
soil = mg.add_zeros("soil__depth", at="node")
soil[:] = 1.0 + 0.25 * np.exp(-(z - np.nanmin(z)) / 8.0)

fig, axes = plt.subplots(1, 2, figsize=(11, 4), layout="constrained")
im0 = axes[0].imshow(z.reshape(mg.shape), cmap="terrain", origin="lower")
axes[0].set_title("Synthetic topography")
plt.colorbar(im0, ax=axes[0], label="Elevation (m)")

im1 = axes[1].imshow(soil.reshape(mg.shape), cmap="viridis", origin="lower")
axes[1].set_title("Soil depth")
plt.colorbar(im1, ax=axes[1], label="Soil depth (m)")
plt.show()

## Route flow

Runout uses the hill-flow receiver fields created by `PriorityFloodFlowRouter` when `separate_hill_flow=True`. These fields are required only when runout is enabled, but routing first is a good habit when coupling landsliding to sediment redistribution.


In [ ]:
router = PriorityFloodFlowRouter(
    mg,
    flow_metric="D8",
    separate_hill_flow=True,
    depression_handler="fill",
    update_hill_depressions=True,
    accumulate_flow=True,
)
router.run_one_step()

required_runout_fields = ("hill_flow__receiver_node", "hill_flow__receiver_proportions")
print({name: mg.at_node[name].shape for name in required_runout_fields})

## Add earthquake PGA forcing

In [ ]:
pga_h = mg.add_zeros("earthquake__horizontal_pga", at="node", clobber=True)
pga_v = mg.add_zeros("earthquake__vertical_pga", at="node", clobber=True)

pga_h[:] = 1.2
pga_v[:] = 0.0
pga_h[mg.boundary_nodes] = np.nan
pga_v[mg.boundary_nodes] = np.nan

fig, ax = plt.subplots(figsize=(5, 4), layout="constrained")
im = ax.imshow(pga_h.reshape(mg.shape), cmap="magma", origin="lower")
ax.set_title("Horizontal PGA")
plt.colorbar(im, ax=ax, label="PGA (g)")
plt.show()

## Run ShallowLandslider without measured-data splitting

Set `split_by_width_config=None` for a physics-only run. This computes stability, unstable regions, aspect subgroups, selected landslide labels, and Newmark displacement without requiring measured length-width KDE data.


In [ ]:
landslider = ShallowLandslider(
    mg,
    cohesion_eff=1.0,
    angle_int_frict=10.0,
    submerged_soil_proportion=0.5,
    pga_h=pga_h,
    pga_v=pga_v,
    aspect_interval=30,
    selection_method="probabilistic",
    proportion_method="conservative",
    random_seed=5000,
    compute_displacement=True,
    time_shaking=10.0,
    displacement_threshold=0.0,
    split_by_width_config=None,
)
landslider.run_one_step()

selected = mg.at_node["landslide__selected_labels"]
summary = {
    "unstable_nodes": int(np.count_nonzero(mg.at_node["landslide__unstable_mask"])),
    "selected_nodes": int(np.count_nonzero(selected > 0)),
    "selected_groups": int(len(np.unique(selected[selected > 0]))),
}
print(summary)
landslider.results["group_properties"].head()

## Plot model outputs

In [ ]:
labels = np.ma.masked_where(selected.reshape(mg.shape) == 0, selected.reshape(mg.shape))
fos = np.ma.masked_invalid(mg.at_node["landslide__factor_of_safety"].reshape(mg.shape))
disp = np.ma.masked_less_equal(
    mg.at_node["landslide__newmark_displacement"].reshape(mg.shape), 0.0
)

fig, axes = plt.subplots(1, 3, figsize=(15, 4), layout="constrained")

im0 = axes[0].imshow(labels, cmap="tab20", origin="lower")
axes[0].set_title("Selected landslide labels")
plt.colorbar(im0, ax=axes[0], label="Label")

im1 = axes[1].imshow(fos, cmap="magma_r", origin="lower")
axes[1].set_title("Factor of safety")
plt.colorbar(im1, ax=axes[1], label="FoS")

im2 = axes[2].imshow(disp, cmap="plasma", origin="lower")
axes[2].set_title("Positive Newmark displacement")
plt.colorbar(im2, ax=axes[2], label="Displacement (m)")

for ax in axes:
    ax.set_xlabel("Column")
    ax.set_ylabel("Row")
plt.show()

## Plot selected-group distributions

In [ ]:
props = landslider.results["group_properties"]
selected_props = props.loc[props["selected"]].copy()

fig, axes = plt.subplots(1, 3, figsize=(13, 4), layout="constrained")
for ax, column, label in zip(
    axes,
    ["area", "median_slope", "median_elevation"],
    ["Area (m?)", "Median slope (degrees)", "Median elevation (m)"],
):
    ax.hist(selected_props[column].dropna(), bins=15, color="0.25", alpha=0.8)
    ax.set_xlabel(label)
    ax.set_ylabel("Count")
    ax.set_title(label)
plt.show()

## Optional runout subcomponent

Runout updates `soil__depth` after Newmark displacement is computed. To run it through `ShallowLandslider`, the grid must already contain `hill_flow__receiver_node` and `hill_flow__receiver_proportions`, and the component must be initialized with `compute_displacement=True`, `enable_runout=True`, and `update_soil=True`.


In [ ]:
soil_before_runout = mg.at_node["soil__depth"].copy()

landslider_runout = ShallowLandslider(
    mg,
    cohesion_eff=1.0,
    angle_int_frict=10.0,
    submerged_soil_proportion=0.5,
    pga_h=pga_h,
    pga_v=pga_v,
    aspect_interval=30,
    selection_method="probabilistic",
    proportion_method="conservative",
    random_seed=5000,
    compute_displacement=True,
    time_shaking=10.0,
    displacement_threshold=0.0,
    enable_runout=True,
    update_soil=True,
    split_by_width_config=None,
)
landslider_runout.run_one_step()

soil_change = mg.at_node["soil__depth"] - soil_before_runout
erosion = landslider_runout._runout._last_erosion
deposition = landslider_runout._runout._last_deposition

print(f"Nodes with changed soil depth: {np.count_nonzero(np.abs(soil_change) > 0):,}")
print(f"Total erosion: {np.sum(erosion):.6g} m-node")
print(f"Total deposition: {np.sum(deposition):.6g} m-node")

## Plot runout diagnostics

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(15, 4), layout="constrained")

for ax, array, title, cmap, label in [
    (
        axes[0],
        np.ma.masked_equal(erosion.reshape(mg.shape), 0.0),
        "Runout erosion",
        "magma",
        "Erosion (m)",
    ),
    (
        axes[1],
        np.ma.masked_equal(deposition.reshape(mg.shape), 0.0),
        "Runout deposition",
        "viridis",
        "Deposition (m)",
    ),
    (
        axes[2],
        np.ma.masked_equal(soil_change.reshape(mg.shape), 0.0),
        "Soil-depth change",
        "coolwarm",
        "Change (m)",
    ),
]:
    im = ax.imshow(array, cmap=cmap, origin="lower")
    ax.set_title(title)
    ax.set_xlabel("Column")
    ax.set_ylabel("Row")
    plt.colorbar(im, ax=ax, label=label)
plt.show()

## Notes

- This notebook intentionally avoids OpenTopography and measured-data KDE inputs so it can run in the Landlab documentation environment.
- For measured-data splitting, pass a populated `split_by_width_config` dictionary to `ShallowLandslider`.
- For runout, keep `PriorityFloodFlowRouter(..., separate_hill_flow=True)` before component initialization.
